# WEEK9 知识精讲：pandas 数据合并、分组与透视表

> **中文说明**：本 notebook 依据 `WEEK9/source/lec8_DataFrame_III.ipynb` 重写。主线是“把多张表拼起来，再按组算指标”：`concat` → `merge` → `groupby` → `pivot_table`。代码全部真实运行，输出已保存；练习答案默认折叠。

**学习顺序**

1. `concat`：上下/左右拼接，重点是索引对齐。
2. `merge`：按 key 做 SQL 式连接，重点是四种 join 和行数变化。
3. `groupby`：split → apply → combine，重点是 `agg` / `filter` / `transform` / `apply` 的区别。
4. `pivot_table`：`groupby + unstack` 的表格化写法。

## 目录

1. 辅助函数与 `concat` 的基本方向
2. 拼接 Series
3. 拼接 DataFrame：axis 与 join
4. 索引冲突：verify_integrity / ignore_index / keys
5. 列名不完全相同时的并集与交集
6. `merge`：inner / left / right / outer
7. 行数怎么变：1:1、1:m、m:m
8. 指定 key：on / left_on / right_on
9. 用索引合并：merge + join
10. 重名列：suffixes
11. `groupby` 基础
12. 分组聚合与 describe
13. `agg`：多函数与字典
14. `filter`：按组过滤
15. `transform`：组内变换
16. `apply`：组内自定义函数
17. 自定义分组键：mapping / function / MultiIndex
18. `pivot_table`：groupby + unstack
19. 多层透视：cut / qcut
20. `aggfunc` 与 `margins`
21. 练习 1：把 Series 追加成一行
22. 练习 2：学生数据合并五步
23. 练习 3：学生 BMI 分组
24. 练习 4：Titanic 生存率透视
25. 速查表与常见坑

## 1. 辅助函数与 `concat` 的基本方向

`np.concatenate` 按**位置**拼接数组；`pd.concat` 按**标签**拼接 pandas 对象。先看清楚方向：

- `axis=0`：上下堆叠，增加行；
- `axis=1`：左右并排，增加列。

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

print("pandas version:", pd.__version__)


def make_df(cols, rows):
    """快速造一个 DataFrame：每个单元格是 列名+行号"""
    data = {c: [str(c) + str(i) for i in rows] for c in cols}
    return pd.DataFrame(data, rows)


print(make_df('ABC', range(3)))

pandas version: 3.0.6
    A   B   C
0  A0  B0  C0
1  A1  B1  C1
2  A2  B2  C2


In [2]:
x = [[1, 2],
     [3, 4]]

print("axis=0（上下）:\n", np.concatenate([x, x]))
print("axis=1（左右）:\n", np.concatenate([x, x], axis=1))

axis=0（上下）:
 [[1 2]
 [3 4]
 [1 2]
 [3 4]]
axis=1（左右）:
 [[1 2 1 2]
 [3 4 3 4]]


## 2. 拼接 Series

Series 有索引，`pd.concat` 会按索引对齐；这一点和 NumPy 的“按位置”完全不同。

In [3]:
ser1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
ser2 = pd.Series(['D', 'E', 'F'], index=[4, 5, 6])

print("axis=0（上下）:")
print(pd.concat([ser1, ser2]))

axis=0（上下）:
1    A
2    B
3    C
4    D
5    E
6    F
dtype: str


In [4]:
print("axis=1（左右）:")
print(pd.concat([ser1, ser2], axis=1))

axis=1（左右）:
     0    1
1    A  NaN
2    B  NaN
3    C  NaN
4  NaN    D
5  NaN    E
6  NaN    F


**注意**：`axis=1` 时索引取并集。这里 `ser1` 是 1–3，`ser2` 是 4–6，所以结果有 6 行，彼此错开的位置是 `NaN`。

In [5]:
# 索引有重叠时，会真正“并排”
ser3 = pd.Series(['X', 'Y', 'Z'], index=[2, 3, 4])
print(pd.concat([ser1, ser3], axis=1))

     0    1
1    A  NaN
2    B    X
3    C    Y
4  NaN    Z


## 3. 拼接 DataFrame：axis 与 join

- `axis=0`：把行堆起来，列取**并集**（`join='outer'`，默认）或**交集**（`join='inner'`）；
- `axis=1`：把列并起来，行索引取**并集**或**交集**。

In [6]:
df1 = make_df('AB', [1, 2])
df2 = make_df('AB', [3, 4])

print("df1:\n", df1)
print("\ndf2:\n", df2)

df1:
     A   B
1  A1  B1
2  A2  B2

df2:
     A   B
3  A3  B3
4  A4  B4


In [7]:
print("axis=0（上下堆叠）:")
print(pd.concat([df1, df2]))

axis=0（上下堆叠）:
    A   B
1  A1  B1
2  A2  B2
3  A3  B3
4  A4  B4


In [8]:
print("axis=1（左右并排）:")
print(pd.concat([df1, df2], axis=1))

axis=1（左右并排）:
     A    B    A    B
1   A1   B1  NaN  NaN
2   A2   B2  NaN  NaN
3  NaN  NaN   A3   B3
4  NaN  NaN   A4   B4


`df1` 的索引是 1、2，`df2` 是 3、4，没有交集，所以 `axis=1, join='inner'` 会得到空表。换成有重叠索引的例子更直观。

In [9]:
df3 = make_df('AB', [2, 3])

print("df1:\n", df1)
print("\ndf3:\n", df3)
print("\naxis=1, join='inner':")
print(pd.concat([df1, df3], axis=1, join='inner'))

df1:
     A   B
1  A1  B1
2  A2  B2

df3:
     A   B
2  A2  B2
3  A3  B3

axis=1, join='inner':
    A   B   A   B
2  A2  B2  A2  B2


**关键**：`join` 在不同 axis 下含义不同：

| 写法 | `join='outer'` | `join='inner'` |
|---|---|---|
| `axis=0` | 列取并集，缺的列填 NaN | 只保留公共列 |
| `axis=1` | 行索引取并集 | 只保留公共索引 |

## 4. 索引冲突：verify_integrity / ignore_index / keys

`pd.concat([df1, df1])` 会保留重复索引。很多时候这会让后续 `loc` 出问题。

In [10]:
print(pd.concat([df1, df1]))

    A   B
1  A1  B1
2  A2  B2
1  A1  B1
2  A2  B2


In [11]:
try:
    pd.concat([df1, df1], verify_integrity=True)
except ValueError as e:
    print("verify_integrity=True -> ValueError:", e)

verify_integrity=True -> ValueError: Indexes have overlapping values: Index([1, 2], dtype='int64')


In [12]:
print("保留原索引:")
print(pd.concat([df2, df1]))
print("\nignore_index=True:")
print(pd.concat([df2, df1], ignore_index=True))

保留原索引:
    A   B
3  A3  B3
4  A4  B4
1  A1  B1
2  A2  B2

ignore_index=True:
    A   B
0  A3  B3
1  A4  B4
2  A1  B1
3  A2  B2


`keys` 可以给每一段加一个来源标签，结果会多一层索引。

In [13]:
df4 = pd.concat([df1, df1], keys=['df1', 'df3'])
print(df4)
print("\nindex:")
print(df4.index)

        A   B
df1 1  A1  B1
    2  A2  B2
df3 1  A1  B1
    2  A2  B2

index:
MultiIndex([('df1', 1),
            ('df1', 2),
            ('df3', 1),
            ('df3', 2)],
           )


## 5. 列名不完全相同时的并集与交集

`axis=0` 时，`join='outer'` 取列并集；`join='inner'` 只保留两边都有的列。

In [14]:
df5 = make_df('ABC', [1, 2])
df6 = make_df('BCD', [3, 4])

print("df5:\n", df5)
print("\ndf6:\n", df6)

df5:
     A   B   C
1  A1  B1  C1
2  A2  B2  C2

df6:
     B   C   D
3  B3  C3  D3
4  B4  C4  D4


In [15]:
print("并集（默认 outer）:")
print(pd.concat([df5, df6]))

并集（默认 outer）:
     A   B   C    D
1   A1  B1  C1  NaN
2   A2  B2  C2  NaN
3  NaN  B3  C3   D3
4  NaN  B4  C4   D4


In [16]:
print("交集（inner）:")
print(pd.concat([df5, df6], join='inner'))

交集（inner）:
    B   C
1  B1  C1
2  B2  C2
3  B3  C3
4  B4  C4


**注意**：`axis=0` 的 `join` 控制的是**列**，不是行。行永远是把两边的行都放进去。

## 6. `merge`：inner / left / right / outer

`merge` 是 SQL 风格的连接：按 key 把两张表的列合到一起。

| `how` | 保留哪些行 |
|---|---|
| `'inner'` | 两边 key 都匹配上的行（默认） |
| `'left'` | 左表全部行 + 右表匹配的行 |
| `'right'` | 右表全部行 + 左表匹配的行 |
| `'outer'` | 两边全部行，匹配不上的填 NaN |

In [17]:
left = pd.DataFrame({
    'key': ['K0', 'K1', 'K2', 'K3'],
    'A': ['A0', 'A1', 'A2', 'A3'],
    'B': ['B0', 'B1', 'B2', 'B3']
})
right = pd.DataFrame({
    'key': ['K0', 'K1', 'K4', 'K5'],
    'C': ['C0', 'C1', 'C4', 'C5'],
    'D': ['D0', 'D1', 'D4', 'D5']
})

print("left:\n", left)
print("\nright:\n", right)

left:
   key   A   B
0  K0  A0  B0
1  K1  A1  B1
2  K2  A2  B2
3  K3  A3  B3

right:
   key   C   D
0  K0  C0  D0
1  K1  C1  D1
2  K4  C4  D4
3  K5  C5  D5


In [18]:
for how in ['inner', 'left', 'right', 'outer']:
    print(f"===== how='{how}' =====")
    print(pd.merge(left, right, how=how))
    print()

===== how='inner' =====
  key   A   B   C   D
0  K0  A0  B0  C0  D0
1  K1  A1  B1  C1  D1

===== how='left' =====
  key   A   B    C    D
0  K0  A0  B0   C0   D0
1  K1  A1  B1   C1   D1
2  K2  A2  B2  NaN  NaN
3  K3  A3  B3  NaN  NaN

===== how='right' =====
  key    A    B   C   D
0  K0   A0   B0  C0  D0
1  K1   A1   B1  C1  D1
2  K4  NaN  NaN  C4  D4
3  K5  NaN  NaN  C5  D5

===== how='outer' =====
  key    A    B    C    D
0  K0   A0   B0   C0   D0
1  K1   A1   B1   C1   D1
2  K2   A2   B2  NaN  NaN
3  K3   A3   B3  NaN  NaN
4  K4  NaN  NaN   C4   D4
5  K5  NaN  NaN   C5   D5



**怎么看**：
- `inner` 只留下 K0、K1；
- `left` 保留 K0–K3，K2/K3 的 C、D 是 NaN；
- `right` 保留 K0、K1、K4、K5；
- `outer` 是两边并集，K2/K3 和 K4/K5 都会有 NaN。

## 7. 行数怎么变：1:1、1:m、m:m

`merge` 后的行数不是简单的“左表行数”或“右表行数”，而取决于 key 的重复情况。

| 关系 | 结果行数 |
|---|---|
| 1:1 | 匹配上的 key 数 |
| 1:m / m:1 | 每个 key 展开成多行 |
| m:m | 两边重复数的乘积（可能行数爆炸） |

In [19]:
# 1:1
emp = pd.DataFrame({
    'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
    'group': ['Accounting', 'Engineering', 'Engineering', 'HR']
})
hire = pd.DataFrame({
    'employee': ['Lisa', 'Bob', 'Jake', 'Sue'],
    'hire_date': [2004, 2008, 2012, 2014]
})

print("1:1 merge:")
print(pd.merge(emp, hire, on='employee'))

1:1 merge:
  employee        group  hire_date
0      Bob   Accounting       2008
1     Jake  Engineering       2012
2     Lisa  Engineering       2004
3      Sue           HR       2014


In [20]:
# m:1：多名员工对应同一个 group，group 信息会重复
supervisor = pd.DataFrame({
    'group': ['Accounting', 'Engineering', 'HR'],
    'supervisor': ['Carly', 'Guido', 'Steve']
})

print("m:1 merge:")
print(pd.merge(emp, supervisor, on='group'))

m:1 merge:
  employee        group supervisor
0      Bob   Accounting      Carly
1     Jake  Engineering      Guido
2     Lisa  Engineering      Guido
3      Sue           HR      Steve


In [21]:
# m:m：每个 group 有多个 skill，每个员工会被展开成多行
skills = pd.DataFrame({
    'group': ['Accounting', 'Accounting', 'Engineering',
              'Engineering', 'HR', 'HR'],
    'skills': ['math', 'spreadsheets', 'coding', 'linux',
               'spreadsheets', 'organization']
})

print("m:m merge:")
print(pd.merge(emp, skills, on='group'))

m:m merge:
  employee        group        skills
0      Bob   Accounting          math
1      Bob   Accounting  spreadsheets
2     Jake  Engineering        coding
3     Jake  Engineering         linux
4     Lisa  Engineering        coding
5     Lisa  Engineering         linux
6      Sue           HR  spreadsheets
7      Sue           HR  organization


**注意**：m:m 连接最容易让行数“爆炸”。合并后先看 `shape`，确认行数符合预期，再继续分析。

## 8. 指定 key：on / left_on / right_on

- key 列名相同：`on='employee'`；
- key 列名不同：`left_on='employee', right_on='name'`；
- 合并后如果不需要右表的 key 列，记得 `drop`。

In [22]:
salary = pd.DataFrame({
    'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
    'salary': [70000, 80000, 120000, 90000]
})

print(pd.merge(emp, salary, left_on='employee', right_on='name'))

  employee        group  name  salary
0      Bob   Accounting   Bob   70000
1     Jake  Engineering  Jake   80000
2     Lisa  Engineering  Lisa  120000
3      Sue           HR   Sue   90000


In [23]:
print(pd.merge(emp, salary,
               left_on='employee', right_on='name').drop(columns='name'))

  employee        group  salary
0      Bob   Accounting   70000
1     Jake  Engineering   80000
2     Lisa  Engineering  120000
3      Sue           HR   90000


## 9. 用索引合并：merge + join

如果 key 已经设为索引，可以用 `left_index=True` / `right_index=True`。

In [24]:
emp_idx = emp.set_index('employee')
hire_idx = hire.set_index('employee')

print("emp_idx:\n", emp_idx)
print("\nhire_idx:\n", hire_idx)

emp_idx:
                 group
employee             
Bob        Accounting
Jake      Engineering
Lisa      Engineering
Sue                HR

hire_idx:
           hire_date
employee           
Lisa           2004
Bob            2008
Jake           2012
Sue            2014


In [25]:
print("pd.merge(..., left_index=True, right_index=True):")
print(pd.merge(emp_idx, hire_idx, left_index=True, right_index=True))

pd.merge(..., left_index=True, right_index=True):
                group  hire_date
employee                        
Bob        Accounting       2008
Jake      Engineering       2012
Lisa      Engineering       2004
Sue                HR       2014


In [26]:
print("emp_idx.join(hire_idx):")
print(emp_idx.join(hire_idx))

emp_idx.join(hire_idx):
                group  hire_date
employee                        
Bob        Accounting       2008
Jake      Engineering       2012
Lisa      Engineering       2004
Sue                HR       2014


`join` 默认就是按索引左连接；`merge` 更通用，可以混合“索引 + 列”作为 key。

In [27]:
print(pd.merge(emp_idx, salary, left_index=True, right_on='name'))

         group  name  salary
0   Accounting   Bob   70000
1  Engineering  Jake   80000
2  Engineering  Lisa  120000
3           HR   Sue   90000


## 10. 重名列：suffixes

两边都有同名列时，pandas 自动加 `_x` / `_y`；可以用 `suffixes` 自定义。

In [28]:
df8 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [1, 2, 3, 4]})
df9 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [3, 1, 4, 2]})

print(pd.merge(df8, df9, on='name'))
print()
print(pd.merge(df8, df9, on='name', suffixes=['_L', '_R']))

   name  rank_x  rank_y
0   Bob       1       3
1  Jake       2       1
2  Lisa       3       4
3   Sue       4       2

   name  rank_L  rank_R
0   Bob       1       3
1  Jake       2       1
2  Lisa       3       4
3   Sue       4       2


## 11. `groupby` 基础

`groupby` 是“拆分—应用—合并”：

1. 按 key 拆成若干组；
2. 每组做聚合/变换/过滤；
3. 合并结果。

`groupby` 本身是惰性的：创建 `DataFrameGroupBy` 对象时不会计算，调用 `.sum()`、`.agg()` 等才出结果。

In [29]:
rng = np.random.default_rng(1)

df = pd.DataFrame({
    'key': ['A', 'B', 'C', 'A', 'B', 'C'],
    'data': range(6),
    'random': rng.random(6)
})
print(df)

  key  data    random
0   A     0  0.511822
1   B     1  0.950464
2   C     2  0.144160
3   A     3  0.948649
4   B     4  0.311831
5   C     5  0.423326


In [30]:
gp = df.groupby('key')
print(gp)
print()
print("sum:")
print(gp.sum(numeric_only=True))


sum:
     data    random
key                
A       3  1.460471
B       5  1.262295
C       7  0.567486


In [31]:
print("只对 random 求和:")
print(df.groupby('key')['random'].sum())
print("\n每组最小值:")
print(df.groupby('key').min(numeric_only=True))

只对 random 求和:
key
A    1.460471
B    1.262295
C    0.567486
Name: random, dtype: float64

每组最小值:
     data    random
key                
A       0  0.511822
B       1  0.311831
C       2  0.144160


In [32]:
print("每组的描述统计:")
print(df.groupby('key')['random'].describe())

每组的描述统计:


     count      mean       std       min       25%       50%       75%       max
key                                                                             
A      2.0  0.730236  0.308884  0.511822  0.621029  0.730236  0.839442  0.948649
B      2.0  0.631148  0.451581  0.311831  0.471490  0.631148  0.790806  0.950464
C      2.0  0.283743  0.197401  0.144160  0.213951  0.283743  0.353535  0.423326


## 12. 分组聚合与 describe

`describe()` 一次给出 count / mean / std / min / 25% / 50% / 75% / max。

In [33]:
print(df.groupby('key').describe())

     data                                          random                                                              \
    count mean      std  min   25%  50%   75%  max  count      mean       std       min       25%       50%       75%   
key                                                                                                                     
A     2.0  1.5  2.12132  0.0  0.75  1.5  2.25  3.0    2.0  0.730236  0.308884  0.511822  0.621029  0.730236  0.839442   
B     2.0  2.5  2.12132  1.0  1.75  2.5  3.25  4.0    2.0  0.631148  0.451581  0.311831  0.471490  0.631148  0.790806   
C     2.0  3.5  2.12132  2.0  2.75  3.5  4.25  5.0    2.0  0.283743  0.197401  0.144160  0.213951  0.283743  0.353535   

               
          max  
key            
A    0.948649  
B    0.950464  
C    0.423326  


## 13. `agg`：多函数与字典

`agg` 可以：

- 对一列给多个函数：`agg(['min','median','max'])`；
- 对不同列给不同函数：`agg({'data':'min', 'random':['min','max']})`；
- 用命名聚合让列名更清晰：`agg(total=('data','sum'))`。

In [34]:
print(df.groupby('key').aggregate(['min', 'median', 'max']))

    data               random                    
     min median max       min    median       max
key                                              
A      0    1.5   3  0.511822  0.730236  0.948649
B      1    2.5   4  0.311831  0.631148  0.950464
C      2    3.5   5  0.144160  0.283743  0.423326

In [35]:
print(df.groupby('key').aggregate({
    'data': 'min',
    'random': ['min', 'max']
}))

    data    random          
     min       min       max
key                         
A      0  0.511822  0.948649
B      1  0.311831  0.950464
C      2  0.144160  0.423326


In [36]:
# 命名聚合
print(df.groupby('key').agg(
    total_data=('data', 'sum'),
    mean_random=('random', 'mean'),
    max_random=('random', 'max')
))

     total_data  mean_random  max_random
key                                     
A             3     0.730236    0.948649
B             5     0.631148    0.950464
C             7     0.283743    0.423326


**注意**：用列表或字典聚合后，列可能变成 MultiIndex。取列时用 tuple，例如 `result[('random','max')]`。

In [37]:
result = df.groupby('key').aggregate({'data': 'min', 'random': ['min', 'max']})
print(result.columns)
print()
print(result[('random', 'max')])

MultiIndex([(  'data', 'min'),
            ('random', 'min'),
            ('random', 'max')],
           )

key
A    0.948649
B    0.950464
C    0.423326
Name: (random, max), dtype: float64


## 14. `filter`：按组过滤

`groupby.filter(func)` 的 `func` 接收**一个组**，返回 `True/False`；返回 `False` 的组会被整组删除。

它和 `df.filter()` 完全不是一回事：后者是按列名/行标签筛选。

In [38]:
def filter_func(group):
    return group['random'].min() < 0.3

print("保留 random 最小值 < 0.3 的组:")
print(df.groupby('key').filter(filter_func))

保留 random 最小值 < 0.3 的组:
  key  data    random
2   C     2  0.144160
5   C     5  0.423326


In [39]:
print("df.filter(like='rand') 是筛列名:")
print(df.filter(like='rand'))

df.filter(like='rand') 是筛列名:
     random
0  0.511822
1  0.950464
2  0.144160
3  0.948649
4  0.311831
5  0.423326


**区分**：
- `groupby(...).filter(f)`：按**组**筛选行；
- `df.filter(like=..., regex=...)`：按**标签**筛选列（或行）。

## 15. `transform`：组内变换

`transform` 返回和原表**同形状**的结果：每个位置用“它所在组的聚合值”来替换/计算。

In [40]:
print("每行减去所在组的均值:")
print(df.groupby('key').transform(lambda x: x - x.mean()))

每行减去所在组的均值:
   data    random
0  -1.5 -0.218414
1  -1.5  0.319316
2  -1.5 -0.139583
3   1.5  0.218414
4   1.5 -0.319316
5   1.5  0.139583


In [41]:
# 也可以直接把组均值广播回原表
df['group_mean'] = df.groupby('key')['random'].transform('mean')
df['diff_from_group_mean'] = df['random'] - df['group_mean']
print(df)

  key  data    random  group_mean  diff_from_group_mean
0   A     0  0.511822    0.730236             -0.218414
1   B     1  0.950464    0.631148              0.319316
2   C     2  0.144160    0.283743             -0.139583
3   A     3  0.948649    0.730236              0.218414
4   B     4  0.311831    0.631148             -0.319316
5   C     5  0.423326    0.283743              0.139583


**和 `agg` 的区别**：`agg` 把每组压成一个值；`transform` 把计算结果广播回每一行，行数不变。

## 16. `apply`：组内自定义函数

`apply` 最灵活：函数接收一个子 DataFrame，可以返回标量、Series 或 DataFrame。

> pandas 3 中建议先选好要处理的列，避免分组列参与计算；`group_keys` 控制结果是否带组索引。

In [42]:
def norm_by_data2(group):
    group = group.copy()               # 不修改原表
    group['random'] = group['random'] / group['data'].sum()
    return group

print("group_keys=False:")
print(df.groupby('key', group_keys=False)[['data', 'random']].apply(norm_by_data2))

group_keys=False:
   data    random
0     0  0.170607
1     1  0.190093
2     2  0.020594
3     3  0.316216
4     4  0.062366
5     5  0.060475


In [43]:
print("group_keys=True:")
print(df.groupby('key', group_keys=True)[['data', 'random']].apply(norm_by_data2))

group_keys=True:
       data    random
key                  
A   0     0  0.170607
    3     3  0.316216
B   1     1  0.190093
    4     4  0.062366
C   2     2  0.020594
    5     5  0.060475


**选择建议**：
- 能用内置聚合 → `agg`；
- 需要保持原形状 → `transform`；
- 需要按组返回不同形状/多种结果 → `apply`（最慢，也最容易出索引问题）。

## 17. 自定义分组键：mapping / function / MultiIndex

`groupby` 的 key 可以是列名，也可以是映射、函数或多个 key。

In [44]:
df_key = df.set_index('key')[['data', 'random']]
print(df_key)

     data    random
key                
A       0  0.511822
B       1  0.950464
C       2  0.144160
A       3  0.948649
B       4  0.311831
C       5  0.423326


In [45]:
mapping = {'A': 'vowel', 'B': 'consonant', 'C': 'consonant'}
print(df_key.groupby(mapping).sum())

           data    random
key                      
consonant    12  1.829781
vowel         3  1.460471


In [46]:
print(df_key.groupby(str.lower).mean())

     data    random
key                
a     1.5  0.730236
b     2.5  0.631148
c     3.5  0.283743


In [47]:
print(df_key.groupby([str.lower, mapping]).mean())

               data    random
key key                      
a   vowel       1.5  0.730236
b   consonant   2.5  0.631148
c   consonant   3.5  0.283743


**注意**：`groupby(str.lower)` 作用在**索引**上；如果 key 还是普通列，应先 `set_index`，或者写 `df.groupby(df['key'].str.lower())`。

## 18. `pivot_table`：groupby + unstack

`pivot_table` 把“分组 + 聚合 + 把某一层放到列”三件事一次完成。

| 参数 | 含义 |
|---|---|
| `values` | 要聚合的数值列 |
| `index` | 变成行索引的分组列 |
| `columns` | 变成列的分组列 |
| `aggfunc` | 聚合函数，默认 `'mean'` |
| `fill_value` | 填充缺失值 |
| `margins` | 是否加总计行/列 |

先看手工版：`groupby + mean + unstack`。

> **依赖**：这一节用 `seaborn` 加载 Titanic 数据集；`sns.load_dataset('titanic')` 第一次运行需要联网。没有网络时可以先用本地 CSV 代替。

In [48]:
import seaborn as sns

titanic = sns.load_dataset('titanic')
print(titanic.shape)
print(titanic.head())

(891, 15)
   survived  pclass     sex   age  sibsp  parch     fare embarked  class    who  adult_male deck  embark_town alive  \
0         0       3    male  22.0      1      0   7.2500        S  Third    man        True  NaN  Southampton    no   
1         1       1  female  38.0      1      0  71.2833        C  First  woman       False    C    Cherbourg   yes   
2         1       3  female  26.0      0      0   7.9250        S  Third  woman       False  NaN  Southampton   yes   
3         1       1  female  35.0      1      0  53.1000        S  First  woman       False    C  Southampton   yes   
4         0       3    male  35.0      0      0   8.0500        S  Third    man        True  NaN  Southampton    no   

   alone  
0  False  
1  False  
2   True  
3  False  
4   True  


In [49]:
by_group = titanic.groupby(['sex', 'class'])['survived'].aggregate('mean')
print(by_group)
print("\nunstack 后:")
print(by_group.unstack())

sex     class 
female  First     0.968085
        Second    0.921053
        Third     0.500000
male    First     0.368852
        Second    0.157407
        Third     0.135447
Name: survived, dtype: float64

unstack 后:
class      First    Second     Third
sex                                 
female  0.968085  0.921053  0.500000
male    0.368852  0.157407  0.135447


同样的事情用 `pivot_table` 写：

In [50]:
print(titanic.pivot_table(
    values='survived',
    index='sex',
    columns='class'
))

class      First    Second     Third
sex                                 
female  0.968085  0.921053  0.500000
male    0.368852  0.157407  0.135447


**注意**：`pivot_table` 默认 `aggfunc='mean'`，所以 `survived` 得到的是生存率；如果要人数，写 `aggfunc='sum'`。

## 19. 多层透视：cut / qcut

`pd.cut` 按**固定边界**分箱；`pd.qcut` 按**分位数**分箱（每组样本数尽量相等）。

In [51]:
age = pd.cut(titanic['age'], bins=[0, 18, 60, 100],
             labels=['child', 'adult', 'senior'])
print(age.head())
print(age.value_counts(dropna=False))

0    adult
1    adult
2    adult
3    adult
4    adult
Name: age, dtype: category
Categories (3, str): ['child' < 'adult' < 'senior']
age
adult     553
NaN       177
child     139
senior     22
Name: count, dtype: int64


In [52]:
print(titanic.pivot_table(
    values='survived',
    index=['sex', age],
    columns='class'
))

class             First    Second     Third
sex    age                                 
female child   0.909091  1.000000  0.511628
       adult   0.972222  0.900000  0.413793
       senior  1.000000       NaN  1.000000
male   child   0.800000  0.600000  0.215686
       adult   0.416667  0.061728  0.136364
       senior  0.083333  0.333333  0.000000


In [53]:
fare_category = pd.qcut(titanic['fare'], q=2)
print(fare_category.value_counts())
print("fare 的中位数:", titanic['fare'].quantile(0.5))

fare
(-0.001, 14.454]     447
(14.454, 512.329]    444
Name: count, dtype: int64
fare 的中位数: 14.4542


In [54]:
print(titanic.pivot_table(
    values='survived',
    index=['sex', age],
    columns=[fare_category, 'class']
))

fare          (-0.001, 14.454]                     (14.454, 512.329]                    
class                    First    Second     Third             First    Second     Third
sex    age                                                                              
female child               NaN  1.000000  0.714286          0.909091  1.000000  0.318182
       adult               NaN  0.880000  0.428571          0.972222  0.914286  0.391304
       senior              NaN       NaN  1.000000          1.000000       NaN       NaN
male   child               NaN  0.000000  0.260870          0.800000  0.818182  0.178571
       adult               0.0  0.083333  0.127907          0.437500  0.030303  0.192308
       senior              NaN  0.333333  0.000000          0.083333       NaN       NaN


**怎么看**：
- `pd.cut` 的边界由你指定，适合“0–18 岁、18–60 岁”这种业务分段；
- `pd.qcut` 的边界由数据分位数决定，适合“把样本按金额分成两半”。
- 结果列是 MultiIndex 时，列头会分成两层显示。

## 20. `aggfunc` 与 `margins`

`aggfunc` 可以是字符串、函数、函数列表，也可以是“列名 → 函数”的字典。

In [55]:
print(titanic.pivot_table(
    index='sex',
    columns='class',
    aggfunc={'survived': 'sum', 'fare': 'mean'}
))

              fare                       survived             
class        First     Second      Third    First Second Third
sex                                                           
female  106.125798  21.970121  16.118810       91     70    72
male     67.226127  19.741782  12.661633       45     17    47


In [56]:
print(titanic.pivot_table(
    values='survived',
    index='sex',
    columns='class',
    margins=True
))

class      First    Second     Third       All
sex                                           
female  0.968085  0.921053  0.500000  0.742038
male    0.368852  0.157407  0.135447  0.188908
All     0.629630  0.472826  0.242363  0.383838


**注意**：`margins=True` 会多出 `All` 行/列；如果同时用 `aggfunc` 字典，先确认你想要的“总计”含义。

## 21. 练习 1：把 Series 追加成一行

把 `record` 追加到 `student_data1`，并让行索引从 0 重新开始。

<details>
<summary>点击查看答案</summary>

```python
record_df = record.to_frame().T
combined = pd.concat([student_data1, record_df], ignore_index=True)
```

`Series.to_frame()` 先变成一列，再 `.T` 转成一行；`ignore_index=True` 重排行索引。

</details>

In [57]:
student_data1 = pd.DataFrame({
    'student_id': ['S1', 'S2', 'S3', 'S4', 'S5'],
    'name': ['Danniella Fenton', 'Ryder Storey', 'Bryce Jensen',
             'Ed Bernal', 'Kwame Morin'],
    'marks': [200, 210, 190, 222, 199]
})

record = pd.Series(['S6', 'Scarlette Fisher', 205],
                   index=['student_id', 'name', 'marks'])

record_df = record.to_frame().T
combined = pd.concat([student_data1, record_df], ignore_index=True)
print(combined)

  student_id              name marks
0         S1  Danniella Fenton   200
1         S2      Ryder Storey   210
2         S3      Bryce Jensen   190
3         S4         Ed Bernal   222
4         S5       Kwame Morin   199
5         S6  Scarlette Fisher   205


**注意**：`record.to_frame().T` 后所有列都是 `object` dtype。如果 `marks` 还要做数值计算，用 `pd.to_numeric(combined['marks'])` 转一下。

In [58]:
print(combined.dtypes)
print("\n转换 marks 后:")
combined['marks'] = pd.to_numeric(combined['marks'])
print(combined.dtypes)

student_id    object
name          object
marks         object
dtype: object

转换 marks 后:
student_id    object
name          object
marks          int64
dtype: object


## 22. 练习 2：学生数据合并五步

数据：`student_data1` 有 `studentid / name / history_marks`，`student_data2` 有 `student_id / name / math_marks`。

(1) 按列拼接两个 DataFrame，保存为 `df1`。

(2) 把 `student_data1` 的 `name` 设为行索引，保存为 `index_data1`。

(3) 用姓名合并 `index_data1` 和 `student_data2`，保留所有学生，删除 id 列，保存为 `df2`。

(4) 让 `df2` 的行索引从 0 开始，保存为 `df3`。

(5) 用 0 填充 `df3` 的缺失值，保存为 `df4`。

<details>
<summary>点击查看答案</summary>

```python
df1 = pd.concat([student_data1, student_data2], axis=1)

index_data1 = student_data1.set_index('name')

df2 = (pd.merge(index_data1, student_data2,
                left_index=True, right_on='name', how='outer')
         .drop(columns=['studentid', 'student_id'])
         .reset_index(drop=True))

df3 = df2.reset_index(drop=True)
df4 = df3.fillna(0)
```

</details>

In [59]:
student_data1 = pd.DataFrame({
    'studentid': ['S1', 'S2', 'S3', 'S4', 'S5'],
    'name': ['Dante Morse', 'Ryder Storey', 'Bryce Jensen',
             'Ed Bernal', 'Kwame Morin'],
    'history_marks': [200, 210, 190, 222, 199]
})

student_data2 = pd.DataFrame({
    'student_id': ['S6', 'S4', 'S1', 'S9', 'S10'],
    'name': ['Scarlette Fisher', 'Ed Bernal', 'Dante Morse',
             'Kaiser William', 'Madeeha Preston'],
    'math_marks': [201, 200, 198, 219, 201]
})

print("student_data1:\n", student_data1)
print("\nstudent_data2:\n", student_data2)

student_data1:
   studentid          name  history_marks
0        S1   Dante Morse            200
1        S2  Ryder Storey            210
2        S3  Bryce Jensen            190
3        S4     Ed Bernal            222
4        S5   Kwame Morin            199

student_data2:
   student_id              name  math_marks
0         S6  Scarlette Fisher         201
1         S4         Ed Bernal         200
2         S1       Dante Morse         198
3         S9    Kaiser William         219
4        S10   Madeeha Preston         201


In [60]:
df1 = pd.concat([student_data1, student_data2], axis=1)
print("(1) df1:\n", df1)

(1) df1:
   studentid          name  history_marks student_id              name  math_marks
0        S1   Dante Morse            200         S6  Scarlette Fisher         201
1        S2  Ryder Storey            210         S4         Ed Bernal         200
2        S3  Bryce Jensen            190         S1       Dante Morse         198
3        S4     Ed Bernal            222         S9    Kaiser William         219
4        S5   Kwame Morin            199        S10   Madeeha Preston         201


**说明**：这里两个表都用默认的 0–4 行索引，所以 `axis=1` 会按位置并排；两个 `name` 列会重名。这一步只是练 `concat`，真正按姓名对齐在 (3) 用 `merge` 完成。

In [61]:
index_data1 = student_data1.set_index('name')
print("(2) index_data1:\n", index_data1)

(2) index_data1:
              studentid  history_marks
name                                 
Dante Morse         S1            200
Ryder Storey        S2            210
Bryce Jensen        S3            190
Ed Bernal           S4            222
Kwame Morin         S5            199


In [62]:
df2 = (pd.merge(index_data1, student_data2,
                left_index=True, right_on='name', how='outer')
         .drop(columns=['studentid', 'student_id'])
         .reset_index(drop=True))
print("(3) df2:\n", df2)

(3) df2:
    history_marks              name  math_marks
0          190.0      Bryce Jensen         NaN
1          200.0       Dante Morse       198.0
2          222.0         Ed Bernal       200.0
3            NaN    Kaiser William       219.0
4          199.0       Kwame Morin         NaN
5            NaN   Madeeha Preston       201.0
6          210.0      Ryder Storey         NaN
7            NaN  Scarlette Fisher       201.0


In [63]:
df3 = df2.reset_index(drop=True)
print("(4) df3:\n", df3)

(4) df3:
    history_marks              name  math_marks
0          190.0      Bryce Jensen         NaN
1          200.0       Dante Morse       198.0
2          222.0         Ed Bernal       200.0
3            NaN    Kaiser William       219.0
4          199.0       Kwame Morin         NaN
5            NaN   Madeeha Preston       201.0
6          210.0      Ryder Storey         NaN
7            NaN  Scarlette Fisher       201.0


In [64]:
df4 = df3.fillna(0)
print("(5) df4:\n", df4)

(5) df4:
    history_marks              name  math_marks
0          190.0      Bryce Jensen         0.0
1          200.0       Dante Morse       198.0
2          222.0         Ed Bernal       200.0
3            0.0    Kaiser William       219.0
4          199.0       Kwame Morin         0.0
5            0.0   Madeeha Preston       201.0
6          210.0      Ryder Storey         0.0
7            0.0  Scarlette Fisher       201.0


## 23. 练习 3：学生 BMI 分组

(1) 按 `school_code` 和 `class` 分组，求 `age` 的 mean / min / max。

(2) 只考虑 age ≤ 13 的学生，按 `school_code` 分组，求 BMI 平均值。

BMI = weight / (height/100)^2

<details>
<summary>点击查看答案</summary>

```python
student_data.groupby(['school_code', 'class'])['age'].agg(['mean', 'min', 'max'])

student_data['BMI'] = student_data['weight'] / (student_data['height'] / 100) ** 2
student_data[student_data['age'] <= 13].groupby('school_code')['BMI'].mean()
```

</details>

In [65]:
student_data = pd.DataFrame({
    'school_code': ['s001', 's002', 's003', 's001', 's002', 's004'],
    'class': ['V', 'V', 'VI', 'VI', 'V', 'VI'],
    'name': ['Alberto Franco', 'Gino Mcneill', 'Ryan Parkes',
             'Eesha Hinton', 'Gino Mcneill', 'David Parkes'],
    'date_Of_Birth ': ['15/05/2002', '17/05/2002', '16/02/1999',
                       '25/09/1998', '11/05/2002', '15/09/1997'],
    'age': [12, 12, 13, 13, 14, 12],
    'height': [173, 192, 186, 167, 151, 159],
    'weight': [70, 85, 93, 65, 50, 60],
    'address': ['street1', 'street2', 'street3',
                'street1', 'street2', 'street4']
}, index=['S1', 'S2', 'S3', 'S4', 'S5', 'S6'])

print(student_data)

   school_code class            name date_Of_Birth   age  height  weight  address
S1        s001     V  Alberto Franco     15/05/2002   12     173      70  street1
S2        s002     V    Gino Mcneill     17/05/2002   12     192      85  street2
S3        s003    VI     Ryan Parkes     16/02/1999   13     186      93  street3
S4        s001    VI    Eesha Hinton     25/09/1998   13     167      65  street1
S5        s002     V    Gino Mcneill     11/05/2002   14     151      50  street2
S6        s004    VI    David Parkes     15/09/1997   12     159      60  street4


In [66]:
print("(1) age 的 mean / min / max:")
print(student_data.groupby(['school_code', 'class'])['age'].agg(['mean', 'min', 'max']))

(1) age 的 mean / min / max:
                   mean  min  max
school_code class                
s001        V      12.0   12   12
            VI     13.0   13   13
s002        V      13.0   12   14
s003        VI     13.0   13   13
s004        VI     12.0   12   12


In [67]:
student_data['BMI'] = student_data['weight'] / (student_data['height'] / 100) ** 2
print("(2) age <= 13 的 BMI 平均值:")
print(student_data[student_data['age'] <= 13]
      .groupby('school_code')['BMI']
      .mean())

(2) age <= 13 的 BMI 平均值:
school_code
s001    23.347683
s002    23.057726
s003    26.881720
s004    23.733238
Name: BMI, dtype: float64


## 24. 练习 4：Titanic 生存率透视

按性别、年龄段 `(0,18] (18,40] (40,65] (65,80]` 和舱位，求生存率。

<details>
<summary>点击查看答案</summary>

```python
age_bins = pd.cut(titanic['age'], bins=[0, 18, 40, 65, 80],
                  labels=['0-18', '18-40', '40-65', '65-80'])

titanic.pivot_table(
    values='survived',
    index=['sex', age_bins],
    columns='class',
    aggfunc='mean'
)
```

</details>

In [68]:
age_bins = pd.cut(titanic['age'], bins=[0, 18, 40, 65, 80],
                  labels=['0-18', '18-40', '40-65', '65-80'])

print(titanic.pivot_table(
    values='survived',
    index=['sex', age_bins],
    columns='class',
    aggfunc='mean'
))

class            First    Second     Third
sex    age                                
female 0-18   0.909091  1.000000  0.511628
       18-40  0.979167  0.914894  0.480000
       40-65  0.961538  0.846154  0.111111
male   0-18   0.800000  0.600000  0.215686
       18-40  0.478261  0.063492  0.146199
       40-65  0.282609  0.105263  0.068966
       65-80  0.250000  0.000000  0.000000


**解读示例**：女性 First Class 的生存率最高，男性 Third Class 最低；年龄段的样本量不同，NaN 表示该组合没有可用年龄数据。

## 25. 速查表与常见坑

### 25.1 合并

| 想要 | 写法 |
|---|---|
| 上下堆叠 | `pd.concat([a, b])` |
| 左右并排 | `pd.concat([a, b], axis=1)` |
| 只保留公共列 | `pd.concat([a, b], join='inner')` |
| 重排行索引 | `ignore_index=True` |
| 加来源层 | `keys=['a', 'b']` |
| 检查重复索引 | `verify_integrity=True` |
| SQL 内连接 | `pd.merge(a, b, how='inner')` |
| 左/右/外连接 | `how='left' / 'right' / 'outer'` |
| 不同列名 | `left_on=..., right_on=...` |
| 索引连接 | `left_index=True, right_index=True` |
| 索引 + 列 | `left_index=True, right_on=...` |
| 重名列后缀 | `suffixes=['_L', '_R']` |
| 按索引左连接 | `a.join(b)` |

### 25.2 分组

| 想要 | 写法 |
|---|---|
| 创建分组对象 | `df.groupby('key')` |
| 多列分组 | `df.groupby(['k1', 'k2'])` |
| 聚合 | `.sum()` / `.mean()` / `.agg(...)` |
| 多函数 | `.agg(['min', 'max'])` |
| 不同列不同函数 | `.agg({'A': 'sum', 'B': 'mean'})` |
| 命名聚合 | `.agg(total=('A', 'sum'))` |
| 按组过滤 | `.filter(func)` |
| 组内变换（同形状） | `.transform(func)` |
| 组内自定义 | `.apply(func)` |
| 组大小 | `.size()` |
| 描述统计 | `.describe()` |

### 25.3 透视

| 想要 | 写法 |
|---|---|
| 生存率表 | `pivot_table(values='survived', index='sex', columns='class')` |
| 人数表 | 加 `aggfunc='sum'` |
| 多指标 | `aggfunc={'survived':'sum', 'fare':'mean'}` |
| 填充空值 | `fill_value=0` |
| 加总计 | `margins=True` |
| 固定分箱 | `pd.cut(x, bins=[...], labels=[...])` |
| 等频分箱 | `pd.qcut(x, q=4)` |

### 25.4 常见坑

1. **`concat` 按标签对齐**，不是按位置；`axis=1` 时索引不同会产生 NaN。
2. **`axis=0` 的 `join` 控制列**：`inner` 只保留公共列。
3. **`verify_integrity=True` 遇到重复索引会报错**，不是自动去重。
4. **`merge` 可能让行数变多**：m:m 连接要先检查 `shape`。
5. **`merge` 默认 `how='inner'`**，会丢掉没有匹配的行；需要保留左表就写 `how='left'`。
6. **`join` 默认按索引**，不是按列。
7. **`groupby.filter` 过滤的是组**，`df.filter` 过滤的是标签。
8. **`transform` 返回同形状**，`agg` 返回每组一行。
9. **`apply` 最灵活也最慢**，能用 `agg`/`transform` 就不要用 `apply`。
10. **`pivot_table` 默认 `aggfunc='mean'`**，人数要用 `'sum'`。
11. **`pd.cut` 是固定边界，`pd.qcut` 是等频分箱**。
12. **聚合/透视后列可能是 MultiIndex**，用 tuple 取列。
13. **Titanic 数据集第一次加载需要联网**（`sns.load_dataset('titanic')`）。

---

**Done.** 建议执行 `Kernel → Restart & Run All`。每做完一次合并，先看 `shape`；每做完一次分组，先看 `index` 和列名。